# K-Means Clustering Demo
This notebook demonstrates **K-Means clustering** using a business-friendly example (Customer Segmentation).
- We will use either the Mall Customers dataset (if uploaded) or synthetic blobs.
- We will step through scaling, elbow/silhouette methods, fitting k-means, and interpreting results.
- Finally, we’ll visualize clusters and centroids, and discuss their business meaning.

In [ ]:
# Install required libraries
!pip install -q scikit-learn matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.datasets import make_blobs

sns.set(style="whitegrid")
%matplotlib inline

In [ ]:
# Load Mall Customers dataset (upload) or generate synthetic fallback
from google.colab import files
uploaded = files.upload()  # Upload Mall_Customers.csv or cancel for fallback

if uploaded:
    fname = list(uploaded.keys())[0]
    df = pd.read_csv(fname)
    print("Loaded", fname)
else:
    # Fallback: synthetic blobs for clear clusters
    X, y_true = make_blobs(n_samples=400, centers=4,
                           cluster_std=[1.0, 1.5, 0.6, 1.2],
                           random_state=42)
    df = pd.DataFrame(X, columns=['feature1','feature2'])
    print("Using synthetic blob data as fallback (2D)")

df.head()

In [ ]:
# Select features and scale
if 'Annual Income (k$)' in df.columns:
    features = ['Annual Income (k$)', 'Spending Score (1-100)']
    X = df[features].copy()
else:
    X = df[['feature1','feature2']].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Scatter plot raw features
plt.figure(figsize=(6,5))
plt.scatter(X.values[:,0], X.values[:,1], s=35, alpha=0.7)
plt.xlabel(X.columns[0]); plt.ylabel(X.columns[1])
plt.title("Raw feature scatter (before clustering)")
plt.show()

In [ ]:
# Elbow & silhouette method
inertias, sil_scores = [], []
ks = range(2,11)
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(ks, inertias, '-o')
plt.xlabel('k'); plt.ylabel('Inertia'); plt.title('Elbow method')

plt.subplot(1,2,2)
plt.plot(ks, sil_scores, '-o')
plt.xlabel('k'); plt.ylabel('Silhouette score'); plt.title('Silhouette score')
plt.show()

print("Silhouette scores:", dict(zip(ks, np.round(sil_scores,3))))

In [ ]:
# Fit chosen k
k = 4
km = KMeans(n_clusters=k, random_state=42, n_init=10)
labels = km.fit_predict(X_scaled)
centers_scaled = km.cluster_centers_
centers = scaler.inverse_transform(centers_scaled)

# Attach labels for plotting
plot_df = X.copy()
plot_df['cluster'] = labels

plt.figure(figsize=(7,6))
palette = sns.color_palette('tab10', n_colors=k)
sns.scatterplot(x=plot_df.columns[0], y=plot_df.columns[1], hue='cluster',
                data=plot_df, palette=palette, s=60, alpha=0.8, edgecolor='k')
for i, c in enumerate(centers):
    plt.scatter(c[0], c[1], marker='X', s=200, c=[palette[i]], edgecolor='k')
    plt.text(c[0], c[1], f'  C{i}', fontsize=12, weight='bold')
plt.title(f'KMeans clusters (k={k}) with centroids')
plt.show()

In [ ]:
# Centroid table and cluster sizes
cluster_sizes = pd.Series(labels).value_counts().sort_index()
centroid_df = pd.DataFrame(centers, columns=X.columns)
centroid_df['size'] = cluster_sizes.values
centroid_df.index = [f'Cluster_{i}' for i in range(len(centroid_df))]

centroid_df

In [ ]:
# Silhouette score & sample cluster members
sil = silhouette_score(X_scaled, labels)
print(f"Silhouette score (k={k}):", round(sil, 3))

df_with_clusters = X.copy()
df_with_clusters['cluster'] = labels
df_with_clusters.groupby('cluster').head(5)

In [ ]:
# Optional PCA projection for higher-dimension data
pca = PCA(n_components=2)
proj = pca.fit_transform(X_scaled)
plt.figure(figsize=(7,6))
sns.scatterplot(x=proj[:,0], y=proj[:,1], hue=labels, palette=palette,
                s=60, edgecolor='k')
plt.title('PCA projection of clustered data')
plt.show()

In [ ]:
# Animation: show how centroids move across iterations
from matplotlib.animation import FuncAnimation

# Initialize random centers for demonstration
np.random.seed(42)
k = 3
centers = X_scaled[np.random.choice(len(X_scaled), k, replace=False)]

fig, ax = plt.subplots(figsize=(7,6))

def update(frame):
    global centers
    ax.clear()
    # Assign clusters
    labels = np.argmin(((X_scaled[:, None, :] - centers[None, :, :])**2).sum(axis=2), axis=1)
    # Plot points
    for i in range(k):
        ax.scatter(X_scaled[labels==i,0], X_scaled[labels==i,1], s=30, alpha=0.6, label=f'Cluster {i}')
    # Plot centers
    ax.scatter(centers[:,0], centers[:,1], c='black', marker='X', s=200)
    ax.set_title(f'Iteration {frame+1}')
    # Recompute centers
    new_centers = np.array([X_scaled[labels==i].mean(axis=0) for i in range(k)])
    centers = new_centers

ani = FuncAnimation(fig, update, frames=5, repeat=False)
plt.close()
from IPython.display import HTML
HTML(ani.to_jshtml())